# 🔄 Step 2: Preprocessing with Ground Truth

Extract training data using provided Ground Truth CSV coordinates.

## What This Notebook Does:
- ✅ Load bands from previous notebook
- ✅ Load Ground Truth CSV (GPS coordinates + labels)
- ✅ Convert GPS coordinates → Pixel coordinates
- ✅ Extract spectral values at ground truth locations
- ✅ Calculate NDVI for those locations
- ✅ Export training data to CSV (~100 labeled samples)

---

**Previous:** [01_data_loading.ipynb](01_data_loading.ipynb)  
**Next:** [03_model_training.ipynb](03_model_training.ipynb)

**Note:** This uses supervised learning with sparse ground truth labels from hackathon organizers.

## Load Libraries and Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from rasterio.transform import rowcol

# Load bands and geotransform from previous notebook
print("📂 Loading bands from previous notebook...")
with open('outputs/loaded_bands.pkl', 'rb') as f:
    bands_data = pickle.load(f)

B02 = bands_data['B02']
B03 = bands_data['B03']
B04 = bands_data['B04']
B08 = bands_data['B08']
transform = bands_data['transform']  # For GPS conversion
crs = bands_data['crs']

print(f"✅ Bands loaded! Shape: {bands_data['shape']}")
print(f"✅ Geotransform loaded for coordinate conversion")

## Load Ground Truth CSV

**Ground Truth Labels from Hackathon:**
- Provided by event organizers
- GPS coordinates (Longitude, Latitude)
- Class labels: 1=Seagrass, 2=Water, 3=Sand

We'll use these labeled points to train our CNN model.

In [ ]:
# Load Ground Truth CSV
print("📂 Loading Ground Truth CSV...")
ground_truth = pd.read_csv("Ground_Truth.csv")

print(f"✅ Ground Truth loaded!")
print(f"   Total labeled points: {len(ground_truth)}")
print(f"\nFirst 5 rows:")
print(ground_truth.head())

# Check class distribution
print(f"\nClass Distribution:")
class_counts = ground_truth['Value'].value_counts().sort_index()
for label, count in class_counts.items():
    class_name = {1: 'Seagrass', 2: 'Water', 3: 'Sand'}.get(label, f'Unknown ({label})')
    print(f"   Class {label} ({class_name}): {count} points ({count/len(ground_truth)*100:.1f}%)")

## Convert GPS Coordinates to Pixel Coordinates

Use rasterio's transform to convert (Longitude, Latitude) → (row, col)

In [ ]:
# Convert GPS coordinates (Lon, Lat) to pixel coordinates (row, col)
print("🔄 Converting GPS coordinates to pixel coordinates...")

# rowcol() returns (row, col) from (x=longitude, y=latitude)
rows, cols = rowcol(transform, ground_truth['Longitude'].values, ground_truth['Latitude'].values)

# Convert to integers and check bounds
rows = np.array(rows, dtype=int)
cols = np.array(cols, dtype=int)

# Filter out-of-bounds coordinates
valid_mask = (
    (rows >= 0) & (rows < B02.shape[0]) &
    (cols >= 0) & (cols < B02.shape[1])
)

print(f"✅ Coordinate conversion complete!")
print(f"   Total points: {len(ground_truth)}")
print(f"   Valid points (within image bounds): {valid_mask.sum()}")
print(f"   Out-of-bounds points: {(~valid_mask).sum()}")

# Filter data
ground_truth_valid = ground_truth[valid_mask].copy()
rows_valid = rows[valid_mask]
cols_valid = cols[valid_mask]

print(f"\n✅ Using {len(ground_truth_valid)} valid ground truth points for training")

## Extract Spectral Values & Calculate NDVI

Extract B02, B03, B04, B08 values at each ground truth pixel location

In [ ]:
# Extract spectral values at ground truth locations
print("📊 Extracting spectral values at ground truth locations...")

training_data = []
for i, (r, c) in enumerate(zip(rows_valid, cols_valid)):
    # Extract spectral values
    b02_val = B02[r, c]
    b03_val = B03[r, c]
    b04_val = B04[r, c]
    b08_val = B08[r, c]
    
    # Calculate NDVI
    ndvi_val = (b08_val - b04_val) / (b08_val + b04_val + 1e-10)
    
    # Get label
    label = ground_truth_valid.iloc[i]['Value']
    
    training_data.append({
        'B02': b02_val,
        'B03': b03_val,
        'B04': b04_val,
        'B08': b08_val,
        'NDVI': ndvi_val,
        'Label': int(label)
    })

# Create DataFrame
df = pd.DataFrame(training_data)

print(f"✅ Spectral extraction complete!")
print(f"\nTraining Data Shape: {df.shape}")
print(f"\nFirst 10 rows:")
print(df.head(10))

# Class distribution in training data
class_names_dict = {1: 'Seagrass', 2: 'Water', 3: 'Sand'}
print(f"\nClass Distribution in Training Data:")
for label, count in df['Label'].value_counts().sort_index().items():
    class_name = class_names_dict.get(label, f'Unknown ({label})')
    print(f"   Class {label} ({class_name}): {count} samples ({count/len(df)*100:.1f}%)")

# Export to CSV
df.to_csv("outputs/training_data.csv", index=False)
print("\n✅ Training data exported to 'outputs/training_data.csv'")

# Visualize ground truth locations on image
print("\n📍 Creating visualization of ground truth locations...")
from matplotlib.colors import ListedColormap

# Create a simple RGB composite for background
rgb = np.dstack([B04/B04.max(), B03/B03.max(), B02/B02.max()])
rgb = np.clip(rgb, 0, 1)

plt.figure(figsize=(16, 8))

plt.subplot(1, 2, 1)
plt.imshow(rgb)
plt.title('RGB Composite with Ground Truth Points', fontsize=14, fontweight='bold')
plt.axis('off')

# Plot ground truth points
colors_gt = {1: 'green', 2: 'blue', 3: 'yellow'}
for label in df['Label'].unique():
    mask = df['Label'] == label
    rows_plot = rows_valid[mask]
    cols_plot = cols_valid[mask]
    plt.scatter(cols_plot, rows_plot, c=colors_gt[label], 
               s=50, edgecolor='black', linewidth=1.5, 
               label=f'{class_names_dict[label]} (n={mask.sum()})', alpha=0.8)
plt.legend(loc='upper right', fontsize=11)

plt.subplot(1, 2, 2)
# NDVI map with ground truth overlay
ndvi_full = (B08 - B04) / (B08 + B04 + 1e-10)
im = plt.imshow(ndvi_full, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, label='NDVI')
plt.title('NDVI Map with Ground Truth Points', fontsize=14, fontweight='bold')
plt.axis('off')

for label in df['Label'].unique():
    mask = df['Label'] == label
    rows_plot = rows_valid[mask]
    cols_plot = cols_valid[mask]
    plt.scatter(cols_plot, rows_plot, c=colors_gt[label], 
               s=50, edgecolor='white', linewidth=1.5, 
               label=f'{class_names_dict[label]}', alpha=0.9)
plt.legend(loc='upper right', fontsize=11)

plt.tight_layout()
plt.savefig('outputs/02_ground_truth_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Visualization saved to 'outputs/02_ground_truth_visualization.png'")

# Also save preprocessing data for later use
preprocessing_data = {
    'training_df': df,
    'rows_valid': rows_valid,
    'cols_valid': cols_valid,
    'class_names': class_names_dict,
    'transform': transform,
    'crs': crs
}

with open('outputs/preprocessing_data.pkl', 'wb') as f:
    pickle.dump(preprocessing_data, f)

print("✅ Preprocessing data saved to 'outputs/preprocessing_data.pkl'")
print("\n" + "="*60)
print("✅ PREPROCESSING COMPLETE!")
print("="*60)
print(f"\n📊 Training Dataset Summary:")
print(f"   • Total samples: {len(df)}")
print(f"   • Features: B02, B03, B04, B08, NDVI (5 features)")
print(f"   • Classes: 3 (Seagrass, Water, Sand)")
print(f"   • Data source: Ground Truth CSV from hackathon")
print("\n📌 Next Step: Open 03_model_training.ipynb to train the CNN")